In [ ]:
# --- Imports ---
import os
import numpy as np
import pandas as pd
import librosa
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import f1_score, confusion_matrix, classification_report, accuracy_score


In [ ]:
# --- 1. Setup ---
ROOT = '/kaggle/input/jan-2026-dl-gen-ai-project/messy_mashup'
STEMS_PATH = os.path.join(ROOT, 'genres_stems')
GENRES = ["blues", "classical", "country", "disco", "hiphop", "jazz", "metal", "pop", "reggae", "rock"]


In [ ]:
# --- Feature Extraction ---
def extract_features(song_path):
    y, sr = librosa.load(os.path.join(song_path, 'other.wav'), sr=22050, duration=10)
    tempo, _ = librosa.beat.beat_track(y=y, sr=sr)
    spec_cent = np.mean(librosa.feature.spectral_centroid(y=y, sr=sr))
    zcr = np.mean(librosa.feature.zero_crossing_rate(y))
    rolloff = np.mean(librosa.feature.spectral_rolloff(y=y, sr=sr))
    return [float(tempo), spec_cent, zcr, rolloff]


In [ ]:
# --- Data Preparation ---
data = []

for g in GENRES:
    gp = os.path.join(STEMS_PATH, g)
    songs = [s for s in os.listdir(gp) if os.path.isdir(os.path.join(gp, s))]
    
    for s in songs[:50]:
        data.append({'path': os.path.join(gp, s), 'genre': g})

df = pd.DataFrame(data)

train_df, val_df = train_test_split(df, test_size=0.2, stratify=df['genre'], random_state=42)

print('Train size:', len(train_df))
print('Validation size:', len(val_df))


In [ ]:
# --- Feature Extraction ---
X_train = np.array([extract_features(p) for p in train_df['path']])
y_train = train_df['genre']

X_val = np.array([extract_features(p) for p in val_df['path']])
y_val = val_df['genre']


In [ ]:
# --- Model Training ---
clf = DecisionTreeClassifier(max_depth=5, random_state=42)
clf.fit(X_train, y_train)

# Predictions
y_pred = clf.predict(X_val)

# Metrics
macro_f1 = f1_score(y_val, y_pred, average='macro')
cm = confusion_matrix(y_val, y_pred, labels=GENRES)
cr = classification_report(y_val, y_pred)
accuracy = accuracy_score(y_val, y_pred)

print(f"Validation Macro F1 Score: {macro_f1:.4f}\n")
print("Accuracy:", accuracy)
print("Detailed Classification Report:")
print(cr)


In [ ]:
# --- Confusion Matrix Visualization ---
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', xticklabels=GENRES, yticklabels=GENRES)
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')
plt.show()


In [ ]:
# --- TP, TN, FP, FN Calculation ---
tp = {}
tn = {}
fp = {}
fn = {}

for i, genre in enumerate(GENRES):
    tp[genre] = cm[i, i]
    fn[genre] = sum(cm[i, :]) - tp[genre]
    fp[genre] = sum(cm[:, i]) - tp[genre]
    tn[genre] = np.sum(cm) - (tp[genre] + fn[genre] + fp[genre])

print("\nTP:", tp)
print("\nTN:", tn)
print("\nFP:", fp)
print("\nFN:", fn)
